In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

weather = spark.table("airquality.silver.weather_hourly").select(
    "city", "time_utc", "temperature_c", "humidity_pct"
)

pm25 = spark.table("airquality.silver.pm25_hourly").select(
    "city",
    (F.col("time_utc") + F.expr("INTERVAL 30 MINUTES")).alias("time_utc"),  # midpoint of the PM2.5 hour
    "sensor_id",
    "pm25_ugm3",
)

In [0]:
hourly = (
    weather.join(pm25, on=["city", "time_utc"], how="inner")
           .withColumn("time_local", F.from_utc_timestamp("time_utc", "Asia/Kolkata"))
           .withColumn("local_date", F.to_date("time_local"))
           .withColumn("local_hour", F.hour("time_local"))
           .withColumn("processed_at", F.current_timestamp())
)

display(
    hourly.groupBy("city").agg(
        F.count("*").alias("matched_hours"),
        F.min("time_utc").alias("first_hour"),
        F.max("time_utc").alias("last_hour"),
    )
)

In [0]:
daily = (
    hourly.groupBy("city", "local_date").agg(
        F.count("*").alias("hours"),
        F.round(F.avg("temperature_c"), 1).alias("avg_temp_c"),
        F.round(F.avg("humidity_pct"), 1).alias("avg_humidity_pct"),
        F.round(F.avg("pm25_ugm3"), 1).alias("avg_pm25"),
        F.round(F.max("pm25_ugm3"), 1).alias("max_pm25"),
    )
    .filter(F.col("hours") >= 18)        # quality rule: a day needs at least 18 of 24 hours
    .orderBy("city", "local_date")
)

display(daily)

In [0]:
hourly.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable("airquality.gold.weather_pm25_hourly")

daily.write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("airquality.gold.city_daily_summary")

print("Gold tables saved")

In [0]:
# 1. Correlation: do PM2.5 and weather move together?
display(
    hourly.groupBy("city").agg(
        F.round(F.corr("humidity_pct", "pm25_ugm3"), 2).alias("corr_humidity_pm25"),
        F.round(F.corr("temperature_c", "pm25_ugm3"), 2).alias("corr_temp_pm25"),
        F.count("*").alias("hours"),
    )
)

In [0]:
# 2. Average PM2.5 by humidity level
display(
    hourly.withColumn(
        "humidity_band",
        F.when(F.col("humidity_pct") < 70, "1. below 70%")
         .when(F.col("humidity_pct") < 85, "2. 70–85%")
         .otherwise("3. 85% and above"),
    )
    .groupBy("city", "humidity_band")
    .agg(
        F.round(F.avg("pm25_ugm3"), 1).alias("avg_pm25"),
        F.count("*").alias("hours"),
    )
    .orderBy("city", "humidity_band")
)